In [ ]:
!pip install requests
!pip install nltk

In [ ]:
from unicodedata import name
from urllib import response
import requests

class GoogleMAPService:
    def __init__(self, api_key):
        self.api_key = api_key

    def get_location(self, address):
        # Code to call Google Maps API and return location data
        pass

    def price_level_to_dollar_signs(self, price_level):
        if price_level == "PRICE_LEVEL_UNSPECIFIED":
            return "N/A"
        elif price_level == "PRICE_LEVEL_FREE":
            return "Free"
        elif price_level == "PRICE_LEVEL_INEXPENSIVE":
            return "$"
        elif price_level == "PRICE_LEVEL_MODERATE":
            return "$$"
        elif price_level == "PRICE_LEVEL_EXPENSIVE":
            return "$$$"
        elif price_level == "PRICE_LEVEL_VERY_EXPENSIVE":
            return "$$$$"
        else:
            return "N/A"

    def _parse_places(self, places: list) -> dict:
        """Shared helper to parse a list of place dicts into the results format."""
        results = {}
        for place in places:
            place_name = place.get('displayName', {}).get('text')
            placeId = place.get('id')
            rating = place.get('rating', 'N/A')
            userRatingCount = place.get('userRatingCount', 0)
            typeLabel = place.get('googleMapsTypeLabel', {"text": "N/A"})

            print(f"--- {place_name} ({rating} stars) ---")

            reviews = []
            for review in place.get('reviews', []):
                reviews.append({
                    "rating": review.get('rating', 'N/A'),
                    "text": review.get('text', {}).get('text', '')
                })

            results[placeId] = {
                "name": place_name,
                "place_id": placeId,
                "rating": rating,
                "userRatingCount": userRatingCount,
                "reviews": reviews,
                "reviewSummary": place.get('reviewSummary', "N/A"),
                "location": place.get('location', None),
                "formattedAddress": place.get('formattedAddress', "N/A"),
                "priceLevel": self.price_level_to_dollar_signs(place.get('priceLevel', "N/A")),
                "typeLabel": typeLabel.get('text', "N/A")
            }

        return results

    def search_by_keyword(self, keyword: str, max_results: int = 10) -> dict:
        """
        Search for places by a free-text keyword query, e.g. "Ice Cream Shop in San Francisco".

        Uses the Google Places Text Search API (v1).

        Args:
            keyword:     Free-text search string.
            max_results: Maximum number of results to return (1–20).

        Returns:
            A dict keyed by place_id, matching the shape returned by get_restaurants().
        """
        payload = {
            "textQuery": keyword,
            "maxResultCount": max_results,
        }

        headers = {
            "Content-Type": "application/json",
            "X-Goog-Api-Key": self.api_key,
            "X-Goog-FieldMask": (
                "places.displayName,places.rating,places.reviews,places.id,"
                "places.userRatingCount,places.formattedAddress,places.location,"
                "places.reviewSummary,places.priceLevel,places.googleMapsTypeLabel"
            ),
        }

        url = "https://places.googleapis.com/v1/places:searchText"
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()

        places = response.json().get("places", [])
        return self._parse_places(places)

    def get_restaurants(self, latitude, longitude, radius=1000) -> dict:
        payload = {
            "includedTypes": ["restaurant"],
            "maxResultCount": 10,
            "locationRestriction": {
                "circle": {
                    "center": {"latitude": latitude, "longitude": longitude},
                    "radius": radius
                }
            }
        }

        headers = {
            "Content-Type": "application/json",
            "X-Goog-Api-Key": self.api_key,
            "X-Goog-FieldMask": (
                "places.displayName,places.rating,places.reviews,places.id,"
                "places.userRatingCount,places.formattedAddress,places.location,"
                "places.reviewSummary,places.priceLevel,places.googleMapsTypeLabel"
            ),
        }

        url = "https://places.googleapis.com/v1/places:searchNearby"
        response = requests.post(url, json=payload, headers=headers)
        restaurants = response.json().get('places', [])
        return self._parse_places(restaurants)

To connect to the Google Maps API, I created a Python service file (GoogleMAPService.py) that encapsulates all API interactions.

To use the service, you can import this service and initialize it with API key stored securely using environment variables (google.colab.userdata) or .env.

The function including
- get_restaurants: Fetches nearby restaurant data based on latitude and longitude (that we will get from the user or application layer), returning up to 10 restaurants (as an example) with the necessary fields relevant to the project, such as name, rating, number of user ratings, price level, location, and review summaries.

In [ ]:
# from GoogleMAPService import GoogleMAPService # import the GoogleMAPService from python file
from google.colab import userdata

## Testing the get restaurants from specific location result
latitude = 37.884974679886064
longitude = -122.29913885431155

API_KEY = userdata.get("GOOGLE_MAPS_API_KEY")
googleMAPService = GoogleMAPService(API_KEY)
results = googleMAPService.search_by_keyword("Taco Shop in San Francisco", 5)
print("==Example result in dictionary==")
results

--- Tacos El Patron (4.6 stars) ---
--- Taquería El Farolito (4.5 stars) ---
--- Underdogs Tres (4.4 stars) ---
--- La Taqueria (4.5 stars) ---
--- Taqueria Los Mayas (4.6 stars) ---
==Example result in dictionary==


{'ChIJW-4ApCd_j4AR7woQDYORh5o': {'name': 'Tacos El Patron',
  'place_id': 'ChIJW-4ApCd_j4AR7woQDYORh5o',
  'rating': 4.6,
  'userRatingCount': 1474,
  'reviews': [{'rating': 5,
    'text': 'When I say these are the BEST birria tacos I have EVER had I am not exaggerating. The beef was cooked perfectly and was tender and juicy. Consomme had a lovely umami and rich flavor. The salsas were made in house, cold, and refreshing. My agua fresca was definitely a bit too sweet but that’s probably how it’s supposed to be. Will absolutely be back next time I’m in the Mission district.'},
   {'rating': 5,
    'text': 'I’m born and raised in Boston, and I thought we had some pretty decent Mexican food here, but boy was I wrong. I was visiting SF and a coworker recommended Tacos El Patron to me. I’m glad I made the journey here because I think it’s the best Mexican food I’ve ever had. I think it’s ruined my enjoyment of Mexican food at home, because now I know what I’m missing out on…I will forever c

**OpenAI Integration (OpenAIService.py)**: The system might later use an OpenAI model to generate restaurant recommendations based on data retrieved from the Google Maps API. The restaurant data is first transformed into a summarized textual context, including key attributes such as rating, number of user reviews, price level, and review summaries.

This context is then passed to the OpenAI model, which analyzes the information and selects the top 4 restaurants that best meet the criteria (e.g., high ratings and strong user engagement). The model returns the selected restaurant IDs, which are then mapped back to the original dataset to produce the final recommended results.

In [ ]:
from OpenAIService import OpenAIService

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
openai_service = OpenAIService(OPENAI_API_KEY)

recommended = openai_service.recommend_restaurants(google_places_data=results)

print("---Recommened restaurants---")
for r in recommended:
    print("Name:", r.get("name"))
    print("Rating:", r.get("rating"))
    print("Price:", r.get("priceLevel"))
    print("Address:", r.get("formattedAddress"))
    print("-" * 40)


ModuleNotFoundError: No module named 'OpenAIService'

Heuristic-Based Recommendation (Baseline):
I implement a non-LLM baseline using a rule-based ranking approach on Google Maps data. After retrieving nearby restaurants, then I filter candidates based on minimum rating and review count, then ranks them using a weighted score combining rating and review volume. The top 4 restaurants are returned as recommendations. This baseline serves as a comparison for the LLM-based approach and will be refined through experimentation.

In [ ]:
import math

def compute_score(place):
    rating = place.get("rating", 0)
    review_count = place.get("userRatingCount", 0)

    return 0.7 * rating + 0.3 * math.log1p(review_count)

def recommend_restaurants_no_llm(restaurants):
    filtered = []

    for place in restaurants.values():
        if place.get("rating", 0) < 4.0:
            continue
        if place.get("userRatingCount", 0) < 50:
            continue
        filtered.append(place)

    ranked = sorted(
        filtered,
        key=compute_score,
        reverse=True
    )

    return ranked


recommendation_no_llm = recommend_restaurants_no_llm(restaurants=results)

print("---Recommened restaurants---")
for r in recommendation_no_llm:
    print("Name:", r.get("name"))
    print("Rating:", r.get("rating"))
    print("Price:", r.get("priceLevel"))
    print("Address:", r.get("formattedAddress"))
    print("-" * 40)

---Recommened restaurants---
Name: La Taqueria
Rating: 4.5
Price: $
Address: 2889 Mission St, San Francisco, CA 94110, USA
----------------------------------------
Name: Taquería El Farolito
Rating: 4.5
Price: $
Address: 2779 Mission St, San Francisco, CA 94110, USA
----------------------------------------
Name: Tacos El Patron
Rating: 4.6
Price: $
Address: 1500 S Van Ness Ave #100, San Francisco, CA 94110, USA
----------------------------------------
Name: Taqueria Los Mayas
Rating: 4.6
Price: $$
Address: 331 Clement St, San Francisco, CA 94118, USA
----------------------------------------
Name: Underdogs Tres
Rating: 4.4
Price: $$
Address: 1224 9th Ave, San Francisco, CA 94122, USA
----------------------------------------


In Heuristic-Based Recommendation, we try to apply sentiment analysis to review text and aggregate review-level sentiment into a restaurant-level sentiment score, which is then combined with rating and review count for heuristic ranking.

In [ ]:
import nltk
import pandas as pd
nltk.download("vader_lexicon")

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

def analyze_review_sentiment(text: str) -> dict:
    if not text or not text.strip():
        return {
            "compound": 0.0,
            "label": "neutral"
        }

    scores = sia.polarity_scores(text)
    compound = scores["compound"]

    if compound >= 0.05:
        label = "positive"
    elif compound <= -0.05:
        label = "negative"
    else:
        label = "neutral"

    return {
        "compound": compound,
        "label": label
    }

def aggregate_restaurant_sentiment(reviews: list[dict]) -> dict:
    if not reviews:
        return {
            "sentiment_score": 0.0,
            "positive_count": 0,
            "neutral_count": 0,
            "negative_count": 0,
            "review_count": 0
        }

    compounds = []
    positive_count = 0
    neutral_count = 0
    negative_count = 0

    for review in reviews:
        text = review.get("text", "")
        result = analyze_review_sentiment(text)

        compounds.append(result["compound"])

        if result["label"] == "positive":
            positive_count += 1
        elif result["label"] == "negative":
            negative_count += 1
        else:
            neutral_count += 1

    sentiment_score = sum(compounds) / len(compounds)

    return {
        "sentiment_score": round(sentiment_score, 4),
        "positive_count": positive_count,
        "neutral_count": neutral_count,
        "negative_count": negative_count,
        "review_count": len(reviews)
    }

def add_sentiment_to_restaurants(restaurants: dict) -> dict:
    updated = {}

    for place_id, info in restaurants.items():
        reviews = info.get("reviews", [])
        sentiment_summary = aggregate_restaurant_sentiment(reviews)

        updated[place_id] = {
            **info,
            **sentiment_summary
        }

    return updated

def compute_score_sentiment(place: dict) -> float:
    rating = place.get("rating", 0)
    review_count = place.get("userRatingCount", 0)
    sentiment_score = place.get("sentiment_score", 0)

    return (
        0.5 * rating +
        0.2 * math.log1p(review_count) +
        0.3 * sentiment_score
    )


def recommend_restaurants_no_llm_with_review(restaurants: dict, top_k: int = 4) -> list[dict]:
    restaurants = add_sentiment_to_restaurants(restaurants)

    filtered = []
    for place in restaurants.values():
        if place.get("rating", 0) < 4.0:
            continue
        if place.get("userRatingCount", 0) < 50:
            continue

        place["final_score"] = compute_score_sentiment(place)
        filtered.append(place)

    ranked = sorted(filtered, key=lambda x: x["final_score"], reverse=True)
    return ranked


recommend_restaurants_with_review = recommend_restaurants_no_llm_with_review(results)

for i, r in enumerate(recommend_restaurants_with_review, 1):
    print(f"#{i} {r['name']}")
    print(f"Rating: {r['rating']} ({r['userRatingCount']} reviews)")
    print(f"Price: {r['priceLevel']}")
    print(f"Address: {r['formattedAddress']}")

    summary = r.get("reviewSummary", {}).get("text", {}).get("text", "N/A")
    print(f"Summary: {summary}")

    print(f"Sentiment Score: {r['sentiment_score']}")
    print(f"Positive: {r['positive_count']} | Neutral: {r['neutral_count']} | Negative: {r['negative_count']}")
    print(f"Total Reviews Used: {r['review_count']}")

    print(f"Final Score: {round(r['final_score'], 4)}")
    print("=" * 80)

# df = pd.DataFrame(recommend_restaurants_with_review)
# df = df[["name", "rating", "userRatingCount","reviewSummary", "formattedAddress", "priceLevel", "positive_count", "neutral_count", "sentiment_score", "negative_count", "review_count", "final_score"]]
# df["summary"] = df["reviewSummary"].apply(
#     lambda x: x.get("text", {}).get("text", "N/A") if isinstance(x, dict) else "N/A"
# )

# df_print = df[["name", "rating", "userRatingCount", "formattedAddress", "priceLevel", "positive_count", "neutral_count", "sentiment_score", "negative_count", "review_count", "final_score"]]

# df_print


#1 La Taqueria
Rating: 4.5 (7151 reviews)
Price: $
Address: 2889 Mission St, San Francisco, CA 94110, USA
Summary: People say this Mexican restaurant serves delicious burritos, tacos, and quesadillas, with the carne asada and carnitas being popular choices. They highlight the generous portions, fresh ingredients, and the option to have burritos "dorado style" (grilled). They also like the lively atmosphere, often with live music, and the efficient service.
Sentiment Score: 0.948
Positive: 5 | Neutral: 0 | Negative: 0
Total Reviews Used: 5
Final Score: 4.3094
#2 Taquería El Farolito
Rating: 4.5 (5622 reviews)
Price: $
Address: 2779 Mission St, San Francisco, CA 94110, USA
Summary: People say this Mexican restaurant serves delicious burritos, tacos, and quesadillas with fresh, flavorful fillings and salsas. They highlight the generous portions, affordable prices, and fast service. They also like the casual, lively vibe and the outdoor seating area.
Sentiment Score: 0.9114
Positive: 5 | N

Both approaches produce overlapping results, consistently identifying strong candidates such as Sam’s Log Cabin and Zaytoon Mediterranean Restaurant and Bar. However, differences arise in the remaining selections.

The heuristic method tends to prioritize restaurants with higher review counts, leading to the inclusion of highly popular but slightly lower-rated options (e.g., Picante with a 4.2 rating). In contrast, the LLM-based approach favors restaurants with consistently higher ratings (≥ 4.5), such as Lulu’s Little Kitchen and Wojia Hunan Cuisine.

When incorporating sentiment analysis into the heuristic model, we observe that sentiment can significantly influence the ranking. For example, restaurants such as Picante and Albany Ao Sen are ranked higher due to strong positive sentiment scores, even when their ratings are lower or comparable to other candidates. This indicates that sentiment may disproportionately impact the final score if not properly balanced.

The current thresholds and weighting are initial heuristics and will be refined through iterative experimentation and combined with other API sources to improve contextual recommendations.